## 1. Import & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from clinical_notes_extraction.config import PROJECT_ROOT
from clinical_notes_extraction.utils.eda import (
    capitalize_first_char,
    clean_and_drop_na_values,
    get_notes_with_duplicate_admission_meds,
    histogram_boxplot,
    iqr_outliers
)

## 2. Load Dataset - Medication on Admission

In [ ]:
df = pd.read_parquet(f'{PROJECT_ROOT}/data/datasets/medication_on_admission.parquet')
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


In [ ]:
df.head()

In [ ]:
df.info()

## 3. Data Cleaning

- Check and update columns data types;
- Drop NA values for meds_on_admission column;

In [ ]:
df1 = df.copy()

In [ ]:
df1.info()

In [ ]:
data_types_cols = ['note_id', 'subject_id', 'hadm_id', 'meds_on_admission', 'text']

df1[data_types_cols] = df1[data_types_cols].astype("string")

**Add new column meds_on_admission_length:**

In [ ]:
df1["meds_on_admission_length"] = df1["meds_on_admission"].str.len()

df1

**To ensure we obtain a representative sample from the 'medication on admission' dataset, we need to remove rows with NULL or NA values in the following key columns:**
- meds_on_admission

In [ ]:
# Only drop rows where a specific column is NaN
df2 = df1.dropna(subset=['meds_on_admission'])

In [ ]:
df2.info()

## 4. Univariate analysis

Don't trust the mean alone. Look at the full shape: skew, tails,
zero-inflation, and values that suggest a log transform later.

### 4.1 Univariate Analysis - Numeric Variables

In [ ]:
df2.describe()

In [ ]:
num_cols = ['meds_on_admission_length']

# Check if the list of numerical columns is not empty
if num_cols:
    # 1. Generate standard descriptive statistics (mean, std, min, max, quartiles)
    # .T transposes the table so columns become rows, making it easier to read
    desc = df2[num_cols].describe().T

    # 2. Calculate Skewness for each column to measure distribution asymmetry
    # (Negative = left tail, Positive = right tail, 0 = perfectly symmetric)
    desc["skew"] = df2[num_cols].skew()

    # 3. Calculate Kurtosis to measure the "tailedness" and sharpness of the peak
    # (High kurtosis means more extreme outliers; low means fewer outliers)
    desc["kurtosis"] = df2[num_cols].kurtosis()

    # 4. Count the number of literal zeros (0) in each column
    # (df2 == 0) creates a True/False mask, and .sum() counts the True values
    desc["n_zeros"] = (df2[num_cols] == 0).sum()

    # 5. Display the final comprehensive statistical summary table
    display(desc)

In [ ]:
histogram_boxplot(df2,'meds_on_admission_length', 40)

Decide whether extreme values are errors, rare-but-real, or a separate
subpopulation.

In [ ]:
# Check if the list of numerical columns is not empty
if num_cols:
    rows = []  # Initialize an empty list to store the results for each column

    # Loop through each numerical column specified in the list
    for c in num_cols:
        # Drop missing values (NaNs) temporarily to ensure accurate IQR calculation
        n, lo, hi = iqr_outliers(df2[c].dropna())

        # Calculate the percentage of outliers based on non-null values,
        # then append all metadata to the rows list
        rows.append(
            {
                "column_name": c,
                "number_of_outliers": n,
                "outliers_percentage": (n / df2[c].notna().sum()) * 100,
                "lower": lo,
                "upper": hi,
            }
        )

    # Convert the list of dictionaries into a DataFrame, sort it by outlier percentage
    # in descending order (highest percentage first), and display the final summary table
    display(pd.DataFrame(rows).sort_values("outliers_percentage", ascending=False))

In [ ]:
df2.head()

#### 4.1.1 Clean meds_on_admission column and calculate the meds_on_admission length on the cleaned column

In [ ]:
df2[df2['meds_on_admission_length'] == 1.0]['meds_on_admission'].unique()

In [ ]:
df2[df2['meds_on_admission_length'] == 1.0]['meds_on_admission'].count()

In [ ]:
df2[df2['meds_on_admission_length'] == 2.0]['meds_on_admission'].count()

In [ ]:
df2[df2['meds_on_admission_length'] == 2.0]['meds_on_admission'].unique()

In [ ]:
df2[df2['meds_on_admission_length']==26568.0]

**Note:**
- As you can see in the value above, the note_id `14266723-DS-19` has lots of whitspaces.

In [ ]:
df3 = df2.copy()


**Function - `clean_note(text)`:** Cleans a single text note and returns the processed version (or `None` if it ends up invalid).

1. Returns `None` if the input is not a `str` (NaN, numbers, `None`).
2. Normalizes line endings (`\r\n` and `\r` → `\n`).
3. Strips whitespace from each line.
4. Removes lines made up only of noise — characters `. - / X ? n` (case-insensitive).
5. Collapses multiple spaces/tabs, limits paragraph breaks to a single blank line, and strips the text.
6. Returns `None` if the result is empty or `"na"` (any capitalization); otherwise returns the cleaned text.

**Function `clean_and_drop_column(df, column)`:**
Applies `clean_note` to an entire DataFrame column.

- Creates a `<column>_cleaned` column with the cleaning result.
- Drops (`dropna`) the rows where the note became invalid/empty.
- Returns the resulting DataFrame.

**Notes**
- The noise pattern does not include `_`, so MIMIC-IV de-identification markers (`___`) are not removed.
- Assigning the new column mutates the original DataFrame in place (risk of `SettingWithCopyWarning` on slices).


In [ ]:
# Preprocess meds_on_admission to handle formatting issues and missing values
df3 = clean_and_drop_na_values(df3, 'meds_on_admission')
df3.info()

In [ ]:
df3[df3['meds_on_admission_length'] == 1.0]['meds_on_admission_cleaned'].unique()

In [ ]:
df3[df3['meds_on_admission_length'] == 1.0]['meds_on_admission_cleaned'].count()

In [ ]:
df3[df3['meds_on_admission_length'] == 2.0]['meds_on_admission_cleaned'].unique()

In [ ]:
df3[df3['meds_on_admission_length'] == 2.0]['meds_on_admission_cleaned'].count()

In [ ]:
# Add column meds_on_admission_length
df3['meds_on_admission_cleaned_length'] = df3['meds_on_admission_cleaned'].str.len()
df3.info()


In [ ]:
df3.describe()

In [ ]:
histogram_boxplot(df3, 'meds_on_admission_cleaned_length', 40)

In [ ]:
df3[df3['meds_on_admission_cleaned_length']==2.0]

In [ ]:
df3[df3['meds_on_admission_cleaned_length']==5849.0]

In [ ]:
df3.head()

### 4.2 Univariate Analysis of Categorical Variables

#### 4.2.1 Univariate Analysis of patient notes -> subject_id variable

- subject_id variable corresponds to patient variable

- Top 10 Patients with the Most Notes

In [ ]:
count_notes_per_patient = df3['subject_id'].value_counts()

print(f"---- Top 20 of Patients with most clinical notes ---- \n\n {count_notes_per_patient.head(20)}")

In [ ]:
count_notes_per_patient.describe()

- Percentage of Patients with Multiple Notes

In [ ]:
# 1. Group by patient and count unique notes per patient
# 2. Check if the count is greater than 1 (returns True/False)
# 3. Average the True/False values to get the overall proportion
patient_fraction = (
    df3.groupby("subject_id")["note_id"]
    .nunique()
    .gt(1)
    .mean()
)

# Output the result formatted as a percentage with 1 decimal place
print(f"{patient_fraction:.1%} of patients have more than one note")

- Medication Profile for Patient **13297743**

In [ ]:
df3[df3['subject_id'] == '13297743']['meds_on_admission_cleaned']

- Verification of Duplicate Notes for Patient with the highest number of notes: **13297743**

In [ ]:
is_duplicated = df3[df3['subject_id'] == '13297743']['meds_on_admission_cleaned'].duplicated().any()

print(f"Are there duplicate medications for this patient? {is_duplicated}")

In [ ]:
patient_df = df3[df3['subject_id'] == '13297743']

patient_df

In [ ]:
patient_id = '13297743'

dups = get_notes_with_duplicate_admission_meds(df3, patient_id)

print(f"Notes with duplicate admission meds for patient id number {patient_id}: {dups['note_id'].to_list()}")

In [ ]:
# Use .isin() to filter for multiple note_ids
df3[df3['note_id'].isin(dups['note_id'])]['meds_on_admission_cleaned']

In [ ]:
# Group by patient and count how many times 'meds_on_admission_cleaned' is a duplicate
duplicate_counts = (
    df3.groupby("subject_id")
    .apply(lambda x: x["meds_on_admission_cleaned"].duplicated(keep="first").sum(), include_groups=False)
    .reset_index(name="duplicate_count")
    .sort_values(by="duplicate_count", ascending=False)
)

print(f"Distinct duplicate groups: {len(duplicate_counts)}\n")
print(duplicate_counts.head(20))

- Check duplicated notes for patient **16662316** that contains the highest number of duplicated notes:

In [ ]:
patient_id = '16662316'

dups = get_notes_with_duplicate_admission_meds(df3, patient_id)

print(f"Notes with duplicate admission meds for patient id number {patient_id}: {dups['note_id'].to_list()}")

In [ ]:
df3[df3['note_id'].isin(dups['note_id'])]['meds_on_admission_cleaned']

In [ ]:
patient_id = '12563258'

dups = get_notes_with_duplicate_admission_meds(df3, patient_id)

print(f"Notes with duplicate admission meds for patient id number {patient_id}: {dups['note_id'].to_list()}")

In [ ]:
df3[df3['note_id'].isin(dups['note_id'])][['note_id', 'meds_on_admission_cleaned']]

- As we can see above, the medication on admission of **12563258-DS-64** note is equal to note **12563258-DS-68**.    
    - But in order to remove these kind of duplicates, we need to transform the first character into capital.

In [ ]:
df4 = capitalize_first_char(df3, 'meds_on_admission_cleaned')

df4.head(10)


In [ ]:
df4[df4['note_id'].isin(dups['note_id'])]['meds_on_admission_cleaned']

In [ ]:
# Group by patient and count how many times 'meds_on_admission_cleaned' is a duplicate
duplicate_counts = (
    df4.groupby("subject_id")
    .apply(lambda x: x["meds_on_admission_cleaned"].duplicated(keep="first").sum(), include_groups=False)
    .reset_index(name="duplicate_count")
    .sort_values(by="duplicate_count", ascending=False)
)

print(f"Distinct duplicate notes per patient:\n")
print(duplicate_counts.head(20))

In [ ]:
df5 = df4.drop_duplicates(subset="meds_on_admission_cleaned", keep="first")

df5.head(10)

In [ ]:
# Group by patient and count how many times 'meds_on_admission_cleaned' is a duplicate
duplicate_counts = (
    df5.groupby("subject_id")
    .apply(lambda x: x["meds_on_admission_cleaned"].duplicated(keep="first").sum(), include_groups=False)
    .reset_index(name="duplicate_count")
    .sort_values(by="duplicate_count", ascending=False)
)

print(f"Distinct duplicate notes with per patient:\n")
print(duplicate_counts.head(10))

-------

### 4.3. Data Cleaning and Data Preparation Analysis

In [ ]:
report_dir = f'{PROJECT_ROOT}/reports/eda'
os.makedirs(report_dir, exist_ok=True)

# One row per filter applied above, read straight off the intermediate frames
attrition = pd.DataFrame([
    ("Discharge notes retrieved",            len(df),  "—"),
    ("After removing notes with no section", len(df2), "No 'Medications on Admission' section"),
    ("After text normalisation",             len(df3), "Section reduced to placeholders / null-equivalent text"),
    ("After deduplication discharge notes",      len(df5), "Remove notes containing the same content even if they belong to different patients"),
], columns=["Stage", "Notes retained", "Reason for removal"])

attrition["Notes removed"] = (-attrition["Notes retained"].diff()).fillna(0).astype(int)
attrition["% of initial"] = attrition["Notes retained"] / len(df)

table_3_1 = attrition[["Stage", "Notes retained", "Notes removed", "% of initial", "Reason for removal"]]
table_3_1.to_html(f'{report_dir}/table_3_1_attrition.html', index=False)
table_3_1

In [ ]:
# df3 carries both length columns on the SAME rows, so this compares the effect
# of normalisation alone — not normalisation plus the rows that were dropped.
def describe_length(s):
    n_out, _, hi = iqr_outliers(s.dropna())
    return {
        "Mean": round(s.mean(), 1), "Median": s.median(),
        "Std. dev.": round(s.std(), 1), "Min": s.min(), "Max": s.max(),
        "Skewness": round(s.skew(), 2), "Kurtosis": round(s.kurtosis(), 2),
        "Outliers (n)": n_out, "Outliers (%)": round(100 * n_out / s.notna().sum(), 1),
    }

table_3_2 = pd.DataFrame({
    "Before cleaning": describe_length(df1["meds_on_admission_length"]),
    "After cleaning":  describe_length(df5["meds_on_admission_cleaned_length"]),
}).reset_index(names="Statistic")

table_3_2.to_html(f'{report_dir}/table_3_2_length_stats.html', index=False)
table_3_2

In [ ]:
# Shared bin edges so both rows have the same resolution — otherwise each
# histogram computes its own bins over its own range and the panels are not
# visually comparable.
bins = np.histogram_bin_edges(
    df3["meds_on_admission_length"].dropna(), bins=40
)

fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharex="col")

for (col, label), (ax_hist, ax_box) in zip(
    [("meds_on_admission_length", "Before normalisation"),
     ("meds_on_admission_cleaned_length", "After normalisation")], axes):
    values = df3[col].dropna()

    ax_hist.hist(values, bins=bins, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax_hist.set_ylabel("Number of notes")
    ax_hist.set_title(label, loc="left", fontsize=9)

    sns.boxplot(x=values, ax=ax_box, color="#4C72B0", fliersize=2, width=0.5)
    ax_box.set(xlabel="", yticks=[])

axes[1][0].set_xlabel("Section length (characters)")
axes[1][1].set_xlabel("Section length (characters)")

fig.tight_layout()
fig.savefig(f'{report_dir}/figure_3_1_length_distribution.svg', bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharex="col")

for (col, label), (ax_hist, ax_box) in zip(
    [("meds_on_admission_length", "Before normalisation"),
     ("meds_on_admission_cleaned_length", "After normalisation")], axes):
    values = df3[col].dropna()
    ax_hist.hist(values, bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax_hist.set_ylabel(f"{label}\nNumber of notes")
    sns.boxplot(x=values, ax=ax_box, color="#4C72B0", fliersize=2, width=0.5)
    ax_box.set(xlabel="", yticks=[])

axes[1][0].set_xlabel("Section length (characters)")
axes[1][1].set_xlabel("Section length (characters)")
fig.tight_layout()
fig.savefig(f'{report_dir}/figure_3_1_length_distribution.svg', bbox_inches="tight")
plt.show()

-----

## 5. Create Final Dataset for Medication Admission

In [ ]:
df5.info()

In [ ]:
final_path = f'{PROJECT_ROOT}/data/datasets/structured_extraction'

# Creates the entire folder structure; does nothing if they already exist
os.makedirs(final_path, exist_ok=True)

df5.to_parquet(f'{final_path}/medication_on_admission_final_dataset.parquet')